In [1]:
%load_ext autoreload
%autoreload 2
from transformers import BartForConditionalGeneration, BartTokenizer, pipeline, T5ForConditionalGeneration, T5Tokenizer
from happytransformer import HappyTextToText
from benchmark import ModelBenchmark
from helpers import get_data_from_file
from neuspell import BertChecker
from helpers import process_and_merge_bert

data folder is set to `c:\users\brumda\documents\neuspell\neuspell\../neuspell_data` script


In [2]:
CHECKPOINTS = "./checkpoints/"
corrupt, clean = get_data_from_file('test')
pred_func_simple = lambda model, text: model(text)[0]['generated_text']
pred_func_vennify = lambda model, data: model.generate_text(f"grammar: {data}").text
pred_func_grammarly = lambda model, text: model(f"Fix grammatical errors: {text}")[0]['generated_text']
benchmark = ModelBenchmark(verbose=True)

Model loaded successfully
Model loaded from detect_typo_models/best_model.pt


C:\FIT\bakalarka\detect_typo_model.py:282: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(path))


In [ ]:
model_path = CHECKPOINTS + "grammarly-coedit-large-finetuned"
model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                   local_files_only=True)
tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=2048)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_grammarly, start_idx=666, num_sen=10)

In [ ]:
model_path = CHECKPOINTS + "pszemraj-bart-base-grammar-synthesis-finetuned"
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                     local_files_only=True)
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_simple, start_idx=23745, num_sen=30)

In [ ]:
model_path = CHECKPOINTS + "oliverguhr-spelling-correction-english-base-finetuned"
model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                     local_files_only=True)
tokenizer = BartTokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=2048)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_simple, start_idx=23745, num_sen=30)

In [ ]:
model_path = "./checkpoints/vennify-t5-base-grammar-correction-finetuned"
happy_tt = HappyTextToText(model_type="T5", model_name=model_path)
res = benchmark.get_wrong_words(happy_tt, corrupt, clean, pred_func_vennify, start_idx=666, num_sen=10)

In [ ]:
model_path = CHECKPOINTS + "prithivida-grammar_error_correcter_v1-finetuned"
model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path=model_path,
                                                   local_files_only=True)
tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path=model_path, local_files_only=True)
corrector = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=2048)

res = benchmark.get_wrong_words(corrector, corrupt, clean, pred_func_simple, start_idx=666, num_sen=10)

In [3]:
path = "checkpoints/subwordbert-probwordnoise/finetuned_model"
checker = BertChecker(device="cuda")
checker.from_pretrained(path)


want to get model fromc:\users\brumda\documents\neuspell\neuspell\../neuspell_data\checkpoints/subwordbert-probwordnoise/finetuned_model
loading vocab from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data\checkpoints/subwordbert-probwordnoise/finetuned_model\vocab.pkl
initializing model
loading pretrained weights from path:c:\users\brumda\documents\neuspell\neuspell\../neuspell_data\checkpoints/subwordbert-probwordnoise/finetuned_model
Loading model params from checkpoint dir: c:\users\brumda\documents\neuspell\neuspell\../neuspell_data\checkpoints/subwordbert-probwordnoise/finetuned_model


In [5]:
pred_func = lambda model, data: model.correct_string(data, correct_spaces=True)
res = benchmark.get_wrong_words(checker, corrupt, clean, pred_func, tokenizer=None, start_idx=666, num_sen=2000)

Original: validPatchDMapWithMove = not . null . validPatchDMapWithMove'
Prediction: validPatchDMapWithMove = not . null . validPatchDMapWithMove'
Clean sentence: validPatchDMapWithMove = not . null . validationErrorsForPatchDMapWithMove
----------------------------------------------------------------------------------------------------
{
  "Correct → Incorrect": [],
  "Incorrect → Incorrect": [
    {
      "orig": "validPatchDMapWithMove'",
      "clean": "validationErrorsForPatchDMapWithMove",
      "pred": "validPatchDMapWithMove'"
    }
  ],
  "Incorrect → Correct": []
}
Original: > A Kademlia DHT implemention on go-libp2p
Prediction: > A Kademlia DHT implementation on go-libp2p
Clean sentence: > A Kademlia DHT implementation on go-libp2p
----------------------------------------------------------------------------------------------------
{
  "Correct → Incorrect": [],
  "Incorrect → Incorrect": [],
  "Incorrect → Correct": [
    {
      "orig": "implemention",
      "clean": "implem

KeyboardInterrupt: 

In [12]:
checker.correct_string("identifies") == "identifies"

True